In [8]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots
import os

In [ ]:
simple_moving_average_st_btc_usdt = pd.read_csv("./data/results.csv")
simple_moving_average_st_btc_usdt.head(15)

,timestamp,open,high,low,close,volume,action,short_avg,long_avg
0,1754463600000,114084.92,114300.00,114019.66,114227.91,319.69253,hold,NaN,NaN
1,1754467200000,114227.90,114250.00,113727.57,113948.22,382.35629,hold,NaN,NaN
2,1754470800000,113948.23,114368.31,113906.96,114093.15,340.97337,hold,NaN,NaN
3,1754474400000,114093.15,114226.69,113899.20,114067.20,335.97303,hold,NaN,NaN
4,1754478000000,114067.20,114404.23,114007.85,114226.69,220.99495,hold,NaN,NaN
5,1754481600000,114226.68,114333.59,113809.02,113914.31,306.06988,hold,NaN,NaN
6,1754485200000,113914.31,114390.88,113579.90,114270.09,671.19500,hold,NaN,NaN
7,1754488800000,114270.09,115200.00,113869.88,115187.96,1041.66895,hold,NaN,NaN
8,1754492400000,115187.96,115515.09,114832.38,115413.87,1490.74630,hold,NaN,NaN
9,1754496000000,115413.87,115477.06,114942.18,115132.49,642.38859,hold,NaN,NaN


In [17]:
simple_moving_average_st_btc_usdt.shape

(1000, 9)

In [18]:
simple_moving_average_st_btc_usdt['date'] = pd.to_datetime(simple_moving_average_st_btc_usdt['timestamp'], unit='ms')

In [19]:
#plot candlestick chart with buy and sell signals where buy is green and sell is red and xaxis is date and yaxis is close price
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, subplot_titles=('Candlestick chart', 'Volume'), row_width=[0.2, 0.7])
fig.add_trace(go.Candlestick(x=simple_moving_average_st_btc_usdt['date'],
                             open=simple_moving_average_st_btc_usdt['open'],
                             high=simple_moving_average_st_btc_usdt['high'],
                             low=simple_moving_average_st_btc_usdt['low'],
                             close=simple_moving_average_st_btc_usdt['close'], name='Candlestick'), row=1, col=1)
fig.add_trace(go.Bar(x=simple_moving_average_st_btc_usdt['date'],
                     y=simple_moving_average_st_btc_usdt['volume'], name='Volume'), row=2, col=1)
fig.add_trace(go.Scatter(x=simple_moving_average_st_btc_usdt['date'],
                         y=simple_moving_average_st_btc_usdt['short_avg'], mode='lines', name='SMA Short', line=dict(color='blue', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=simple_moving_average_st_btc_usdt['date'],
                         y=simple_moving_average_st_btc_usdt['long_avg'], mode='lines', name='SMA Long', line=dict(color='orange', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=simple_moving_average_st_btc_usdt[simple_moving_average_st_btc_usdt['action'] == 'buy']['date'],
                         y=simple_moving_average_st_btc_usdt[simple_moving_average_st_btc_usdt['action'] == 'buy']['close'],
                         mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')), row=1, col=1)
fig.add_trace(go.Scatter(x=simple_moving_average_st_btc_usdt[simple_moving_average_st_btc_usdt['action'] == 'sell']['date'],
                         y=simple_moving_average_st_btc_usdt[simple_moving_average_st_btc_usdt['action'] == 'sell']['close'],
                         mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')), row=1, col=1)
fig.update_layout(title='BTC/USDT Simple Moving Average Strategy', yaxis_title='Price (USDT)', xaxis2_title='Date', yaxis2_title='Volume', xaxis_rangeslider_visible=False)
fig.show()

In [24]:
# Create a function to calculate returns
def calculate_returns(df: pd.DataFrame) -> pd.DataFrame:
    temp_df = df.copy()
    temp_df['returns'] = temp_df['close'].pct_change().fillna(0) # Daily returns
    temp_df['strategy_returns'] = temp_df['returns'] * temp_df['action'].shift(1).replace({'buy': 1, 'sell': -1, 'hold': 0}).fillna(0) # Strategy returns
    temp_df['cumulative_returns'] = (1 + temp_df['returns']).cumprod() - 1  # Cumulative returns
    temp_df['cumulative_strategy_returns'] = (1 + temp_df['strategy_returns']).cumprod() - 1 
    return temp_df



simple_moving_average_st_btc_usdt_with_returns = calculate_returns(simple_moving_average_st_btc_usdt)
simple_moving_average_st_btc_usdt_with_returns.tail(10)

/tmp/ipykernel_87724/1572041416.py:5: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,timestamp,open,high,low,close,volume,action,short_avg,long_avg,date,returns,strategy_returns,cumulative_returns,cumulative_strategy_returns
990,1758027600000,115439.36,115558.06,114737.11,115200.00,1064.59506,hold,115376.2356,114322.91345,2025-09-16 13:00:00,-0.002073,-0.0,0.008510,0.00396
991,1758031200000,115200.00,115306.71,114811.00,115296.86,607.41382,hold,115366.6466,114343.83145,2025-09-16 14:00:00,0.000841,0.0,0.009358,0.00396
992,1758034800000,115296.86,115905.89,115010.66,115905.88,653.80057,hold,115377.8022,114366.95890,2025-09-16 15:00:00,0.005282,0.0,0.014690,0.00396
993,1758038400000,115905.88,116068.92,115608.28,116068.06,691.27604,hold,115391.1284,114388.87795,2025-09-16 16:00:00,0.001399,0.0,0.016109,0.00396
994,1758042000000,116068.05,116870.76,116056.99,116461.54,1252.39334,hold,115415.8790,114410.71070,2025-09-16 17:00:00,0.003390,0.0,0.019554,0.00396
995,1758045600000,116461.55,116776.66,116226.12,116418.86,486.09455,hold,115434.6562,114432.56500,2025-09-16 18:00:00,-0.000366,-0.0,0.019181,0.00396
996,1758049200000,116418.87,116820.00,116395.60,116800.12,525.61626,hold,115459.5056,114456.30345,2025-09-16 19:00:00,0.003275,0.0,0.022518,0.00396
997,1758052800000,116800.13,116908.83,116722.56,116839.47,367.21592,hold,115488.5308,114480.32825,2025-09-16 20:00:00,0.000337,0.0,0.022863,0.00396
998,1758056400000,116839.47,116940.26,116584.45,116800.51,295.98351,hold,115511.8412,114504.79805,2025-09-16 21:00:00,-0.000333,-0.0,0.022522,0.00396
999,1758060000000,116800.51,116964.27,116694.73,116780.33,269.76867,hold,115531.4046,114525.47445,2025-09-16 22:00:00,-0.000173,-0.0,0.022345,0.00396


In [21]:
simple_moving_average_st_btc_usdt_with_returns

,timestamp,open,high,low,close,volume,action,short_avg,long_avg,date,returns,strategy_returns,cumulative_returns,cumulative_strategy_returns
0,1754463600000,114084.92,114300.00,114019.66,114227.91,319.69253,hold,NaN,NaN,2025-08-06 07:00:00,0.000000,0.0,0.000000,0.00
1,1754467200000,114227.90,114250.00,113727.57,113948.22,382.35629,hold,NaN,NaN,2025-08-06 08:00:00,-0.002449,0.0,-0.002449,0.00
2,1754470800000,113948.23,114368.31,113906.96,114093.15,340.97337,hold,NaN,NaN,2025-08-06 09:00:00,0.001272,0.0,-0.001180,0.00
3,1754474400000,114093.15,114226.69,113899.20,114067.20,335.97303,hold,NaN,NaN,2025-08-06 10:00:00,-0.000227,0.0,-0.001407,0.00
4,1754478000000,114067.20,114404.23,114007.85,114226.69,220.99495,hold,NaN,NaN,2025-08-06 11:00:00,0.001398,0.0,-0.000011,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1758045600000,116461.55,116776.66,116226.12,116418.86,486.09455,hold,115434.6562,114432.56500,2025-09-16 18:00:00,-0.000366,0.0,0.019181,114385.49
996,1758049200000,116418.87,116820.00,116395.60,116800.12,525.61626,hold,115459.5056,114456.30345,2025-09-16 19:00:00,0.003275,0.0,0.022518,114385.49
997,1758052800000,116800.13,116908.83,116722.56,116839.47,367.21592,hold,115488.5308,114480.32825,2025-09-16 20:00:00,0.000337,0.0,0.022863,114385.49
998,1758056400000,116839.47,116940.26,116584.45,116800.51,295.98351,hold,115511.8412,114504.79805,2025-09-16 21:00:00,-0.000333,0.0,0.022522,114385.49
